In [1]:
import yaml
import os
import threading
import time
from datetime import datetime
import numpy as np

In [2]:
from AccidentError import AccidentError
from NavigationBase import NavigationBase
from DistanceNavigation import DistanceNavigation

In [3]:
import easygopigo3 as easy

my_gpg3 = easy.EasyGoPiGo3(use_mutex=True)

In [4]:
# collect instructions
def get_instructions_from_file(filename):
    drive_path = {"forward": [], "turn": []}
    path_order = []
    with open(filename, 'r') as file:
        drive_path = {"forward": [], "turn": []}
        path_order = []
        data = yaml.safe_load(file)
        steps = data["steps"]
        for i,step in enumerate(steps):
            drive_path[list(step.keys())[0]].append(list(step.values())[0])
            path_order.append(list(step.keys())[0])
    return drive_path, path_order

In [ ]:
def driving_loop(nav_base,direction,dist_or_angle):
    if nav_error.is_set(): return 0
    distance_traveled = 0
    nav_base.robot.reset_encoders(blocking=False)
    if direction == "forward":
        nav_base.forward(dist_or_angle)
        #print("Estimated length of drive section (seconds)",((dist_or_angle/nav_base.wheel_circumference)*360) / nav_base.robot.get_speed())
        target = (dist_or_angle / nav_base.wheel_circumference) * 360
        while not nav_base.robot.target_reached(target, target):
            distance_traveled = (nav_base.robot.read_encoders()[0]/360)*nav_base.wheel_circumference
            if dist_or_angle - distance_traveled >= 1: nav_base.update_distance(dist_or_angle - distance_traveled)
            else: 
                nav_base.stop()
                break
            time.sleep(0.1)
        #time.sleep(((dist_or_angle/nav_base.wheel_circumference)*360) / nav_base.robot.get_speed())
    elif direction == "turn":
        nav_base.turn(dist_or_angle)
        #print("Estimated length of turn (seconds)",((((abs(dist_or_angle)/90)*(2*math.pi*turn_radius))/nav_base.wheel_circumference)*360) / nav_base.robot.get_speed())
        target = (((abs(dist_or_angle)/360)*(2*math.pi*nav_base.turn_radius))/nav_base.wheel_circumference) * 360
        while not nav_base.robot.target_reached(target, target):
            distance_traveled = (nav_base.robot.read_encoders()[0]/360)*nav_base.wheel_circumference
            nav_base.update_distance(((abs(dist_or_angle)/360)*(2*math.pi*nav_base.turn_radius)) - distance_traveled)
            if nav_base.dist_left <= 0.5: 
                nav_base.stop()
                break
            time.sleep(0.1)
        #time.sleep(((((abs(dist_or_angle)/90)*(2*math.pi*nav_base.turn_radius))/nav_base.wheel_circumference)*360) / nav_base.robot.get_speed())
        nav_base.robot.reset_speed()
    print("distance:",distance_traveled)
    return distance_traveled


In [ ]:
def follow_instructions(path, order, nav_base):
    distance_traveled = 0
    for direction in order:
        if nav_error.is_set():
            break
        dist_or_angle = path[direction].pop(0)
        #if direction=="forward": distance_traveled += dist_or_angle
        #elif direction=="turn": distance_traveled += (abs(dist_or_angle)/360)*(2*math.pi*turn_radius)
        distance_traveled += driving_loop(nav_base,direction=direction,dist_or_angle=dist_or_angle)
    nav_base.nav_finished()
    with open(os.path.join(save_dir,f"trial_{trial_num:04d}_results.yml"),"w") as file:
        yaml.dump({"nav_type":"distance","distance_traveled":distance_traveled, "obstacles_encountered": nav_base.obstacle_count, "accidents": nav_base.accidents},file)

In [7]:
def emergency_stop(nav_base):
    try:
        while True:
            nav_base.emergency_stop()
            time.sleep(0.005)
    except AccidentError:
        print("Accident occured")
        nav_error.set()
        nav_base.nav_finished()

In [8]:
def run_navigation(nav_base):
    try:
        while True:
            if nav_error.is_set():
                break
            if nav_base.nav_done:
                break
            nav_base.run_navigation()
    except AccidentError:
        print("Accident occured")
        nav_error.set()
        nav_base.nav_finished()

In [9]:
# Testing drive paths
def main():
    #instruction_dir = "drive_instructions"
    #instruction_files = list(map(lambda file: os.path.join(instruction_dir, file),os.listdir(instruction_dir)))
    global nav_type, camera_ip, trials,trial_num,drive_file,save_dir
    trial_num = None
    config_dict = {}
    with open("trial_config.yml","r") as config_file:
        config_dict = yaml.safe_load(config_file)
        if config_dict["n_trials"] > config_dict["last_trial"]: 
            config_dict["last_trial"] += 1
            nav_type = config_dict["nav_system"]
            camera_ip = config_dict["camera_ip"]
            trials = config_dict["n_trials"]
            trial_num = config_dict["last_trial"]
            drive_file = config_dict["drive_instructions"]
            save_dir = config_dict["save_directory"]
        config_file.close()
    with open("trial_config.yml","w") as config_file:
        yaml.dump(config_dict,config_file)
        config_file.close()

    if trial_num is not None:
        print("Trial", trial_num)
        path,order = get_instructions_from_file(filename=drive_file)

        if nav_type == "distance": nav_base = DistanceNavigation(my_gpg3,camera_ip)
        else: nav_base = NavigationBase(my_gpg3,camera_ip)
        
        global nav_error, ignore_lane, stop_sign
        nav_error = threading.Event()
        ignore_lane = threading.Event()
        stop_sign = threading.Event()
        
        try:
            drive_thread = threading.Thread(target=follow_instructions,args=(path,order,nav_base,),daemon=True)
            drive_thread.start()
            if nav_type is not None:
                nav_thread = threading.Thread(target=run_navigation,args=(nav_base,),daemon=True)
                nav_thread.start()
            else:
                emergency_stop_thread = threading.Thread(target=emergency_stop,args=(nav_base,),daemon=True)
                emergency_stop_thread.start()
        except Exception as e:
            print("Unknown error:",e)
    else:
        print("All trials completed")


In [10]:
main()

300
slowing down
speeding up
distance: 49.37308733333334


In [11]:
# get instructions
# run navigation
#   follow instructions -> forward or turn
#   while following instructions, should check surrounding & do actions based on surrounding
#       ex. check with distance, pathfinding, and potentially nothing
#   benchmark navigation - manage metric collection

In [11]:
for file in os.listdir('test_collection'):
    if os.path.isdir(os.path.join("test_collection",file)): continue
    os.remove(os.path.join("test_collection",file))